In [1]:
from pathlib import Path
import sys

# Move up one level to the project root
root = Path.cwd().parent # Moving the root folder to one parent folder up.
sys.path.append(str(root))


In [3]:
import pandas as pd
import numpy as np
import json

from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [4]:
# Collecting dataframes from Elliptic datasets.
features_df = pd.read_csv("../data/elliptic/elliptic_txs_features.csv", header=None)
classes_df = pd.read_csv("../data/elliptic/elliptic_txs_classes.csv")
edges_df = pd.read_csv("../data/elliptic/elliptic_txs_edgelist.csv")

# Collecting important features.
transaction_ids = features_df.iloc[:, 0].values # First column of features dataframe is the transaction ID.
time_steps = features_df.iloc[:, 1].values # Second column of features dataframe is the discrete time step.
node_features = features_df.iloc[:, 2:].values # The rest of 165 columns are features related to bitcoin transactions.

In [9]:
# Rename columns of Features DataFrame:
# First column is the transaction Id
# Second column is the time step (total of 49 discrete time steps)
# First 94 Features are Bitcoin local anunymous data
# Second 73 Features are Aggregated anonymous data.
features_df.columns = (
    ['txId', 'time_step'] +
    [f'local_{i}' for i in range(1, 94)] +
    [f'agg_{i}'   for i in range(1, 73)]
)

features_df

# Keep only labelled nodes (drop 'unknown')
merged = features_df.merge(classes_df, on='txId')
labelled = merged[merged['class'] != 'unknown'].copy()
labelled['class'] = labelled['class'].map({'1': 1, '2': 0})  # illicit=1, licit=0

labelled[labelled['class'] == 1].head() # Show top 5 Illicit transactions

,txId,time_step,local_1,local_2,local_3,local_4,local_5,local_6,local_7,local_8,...,agg_64,agg_65,agg_66,agg_67,agg_68,agg_69,agg_70,agg_71,agg_72,class
907,232629023,1,-0.172669,0.048298,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.163319,...,-0.285627,0.241128,0.241406,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,1
1361,230389796,1,-0.164417,0.048298,-1.201369,0.178180,-0.043875,0.222447,-0.061584,-0.163580,...,-0.500080,-0.979074,-0.978556,-0.098889,-0.087490,-0.084674,-0.140597,-1.760926,-1.760984,1
2718,17387772,1,-0.164272,0.048298,-1.201369,0.028105,-0.043875,0.054722,-0.061584,-0.163191,...,1.757982,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,1
2815,232947878,1,-0.168195,0.048298,-1.201369,-0.046932,-0.063725,-0.029140,-0.061584,-0.163574,...,-0.600999,0.241128,0.241406,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,1
3423,16754007,1,-0.169109,0.048298,-1.201369,-0.046932,-0.043875,-0.029140,-0.061584,-0.162900,...,0.584799,-0.979074,-0.978556,-0.098889,-0.087490,-0.084674,-0.140597,1.519700,1.521399,1


In [10]:
# Separate Features from the fixed columns (like node_id, class and time_step):
feature_cols = [c for c in labelled.columns
                if c.startswith('local_') or c.startswith('agg_')]

# Feature Nodes
X = labelled[feature_cols]

# Class Nodes
y = labelled['class']



In [11]:
# Random Forest just to calculate the importance of features (no need to tune, just for ranking)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Train the Random Forest.
rf.fit(X, y)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
rf.feature_importances_

array([3.06390264e-03, 6.31049867e-03, 2.20753988e-03, 6.63248454e-03,
       3.74037922e-02, 6.72067929e-03, 5.56645633e-05, 1.15614072e-02,
       3.79630501e-03, 3.56318943e-03, 2.50988619e-03, 1.20892457e-03,
       7.86269175e-04, 3.91431201e-02, 0.00000000e+00, 5.60652395e-03,
       2.47894303e-03, 3.54153158e-02, 2.43934524e-03, 7.39579616e-04,
       9.08678845e-04, 6.05093632e-03, 2.15512032e-02, 1.80145241e-03,
       1.16310603e-02, 1.14388378e-03, 6.55600317e-04, 7.69619845e-03,
       2.14570929e-02, 5.13359921e-03, 1.47854429e-02, 6.92450422e-04,
       1.88787501e-03, 1.97788522e-04, 7.24376545e-04, 1.58380698e-04,
       5.79750304e-04, 7.99365326e-05, 7.36548143e-05, 6.68036121e-03,
       1.96324289e-02, 3.76683827e-03, 2.80637629e-02, 6.11504374e-04,
       4.56348781e-04, 1.26573986e-02, 4.51590036e-02, 2.85287775e-03,
       2.33654813e-02, 5.17820721e-04, 5.13693305e-04, 1.92267862e-02,
       3.69330269e-02, 3.86713781e-03, 3.45876415e-02, 8.02426918e-04,
      

In [13]:
# Top 15 features
importance_series = pd.Series(rf.feature_importances_, index=feature_cols)


In [15]:
# Calculate the 15 most important features after RF training
top15 = importance_series.nlargest(15)

top15

local_47    0.045159
local_90    0.044220
local_14    0.039143
local_5     0.037404
local_53    0.036933
local_18    0.035415
local_55    0.034588
local_43    0.028064
local_60    0.026279
agg_45      0.026091
local_49    0.023365
local_23    0.021551
local_29    0.021457
local_66    0.020542
local_41    0.019632
dtype: float64

In [21]:
KEY_FEATURES = top15.index.to_list()

KEY_FEATURES_LABELS = [
    f"{ 'Local' if feature.startswith('local') else 'Aggregate' } F{ feature.split('_')[1] }"
    for feature in KEY_FEATURES
]

KEY_FEATURES_LABELS

['Local F47',
 'Local F90',
 'Local F14',
 'Local F5',
 'Local F53',
 'Local F18',
 'Local F55',
 'Local F43',
 'Local F60',
 'Aggregate F45',
 'Local F49',
 'Local F23',
 'Local F29',
 'Local F66',
 'Local F41']

In [26]:
# Confirm class distribution
print(classes_df['class'].value_counts())


class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64
